# VocalCoachTCN — Model Visualization

Visualizes the TCN backbone used for Stage 1 (`tcn`, hidden=256, 8 blocks, 4 attention layers).

**Note on `visualtorch.graph_view`:** it traces the full autograd graph recursively and does **not** scale to a model this deep (8 TCN blocks + 4 attention layers = hundreds of nested ops → the recursion effectively hangs, which is the `KeyboardInterrupt` seen previously). Use `torchinfo.summary` for the layer/param breakdown and `visualtorch.layered_view` for a block diagram instead.

In [2]:
import os
import sys

# Visualization is CPU-only: visualtorch/torchinfo feed CPU tensors, and any
# stray CUDA placement causes a device-mismatch crash. Hide the GPU before
# importing torch so everything stays on CPU.
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, os.path.abspath('..'))
from vocalcoach.model import build_model

In [3]:
# tcn(hidden=256, blocks=8, causal=False, attn=4×4h) — the Stage-1 backbone
# Force CPU: visualtorch feeds CPU input tensors, and build_model may place the
# model on CUDA — a device mismatch otherwise crashes the viz calls below.
model = build_model(
    arch='tcn',
    hidden=256,
    n_blocks=8,
    causal=False,
    n_attn_layers=4,
).eval().cpu()

# Input is (batch, time, n_mels=40) — NOT an image. T can be any length; 128 frames = ~1.28 s.
INPUT_SHAPE = (1, 128, 40)

VocalCoachTCN: 7,775,854 parameters (hidden=256, blocks=8, causal=False, attn=4×4h)


## 1. Layer / parameter summary (torchinfo)

The most useful view for a deep multi-head model: per-layer output shapes and parameter counts, including the VAD / pitch / technique / note heads.

In [4]:
from torchinfo import summary

summary(
    model,
    input_size=INPUT_SHAPE,
    col_names=('input_size', 'output_size', 'num_params'),
    depth=3,
    row_settings=('var_names',),
)

Layer (type (var_name))                       Input Shape               Output Shape              Param #
VocalCoachTCN (VocalCoachTCN)                 [1, 128, 40]              [1, 128, 1]               --
├─Conv1d (input_proj)                         [1, 40, 128]              [1, 256, 128]             10,496
├─ModuleList (blocks)                         --                        --                        --
│    └─TCNBlock (0)                           [1, 128, 256]             [1, 128, 256]             --
│    │    └─Conv1d (conv)                     [1, 256, 130]             [1, 256, 128]             196,864
│    │    └─LayerNorm (norm)                  [1, 128, 256]             [1, 128, 256]             512
│    │    └─GELU (act)                        [1, 128, 256]             [1, 128, 256]             --
│    │    └─Dropout (dropout)                 [1, 128, 256]             [1, 128, 256]             --
│    └─TCNBlock (1)                           [1, 128, 256]             [1, 

## 2. Layered block diagram (visualtorch)

`layered_view` walks `model.modules()` instead of the autograd graph, so it renders quickly even for deep nets. Pass the **correct** audio input shape `(1, 128, 40)` — the earlier `(1, 3, 224, 224)` was an image shape and does not match this model.

In [5]:
import visualtorch

img = visualtorch.layered_view(model, input_shape=INPUT_SHAPE, draw_volume=False)

plt.figure(figsize=(14, 8))
plt.axis('off')
plt.tight_layout()
plt.imshow(img)
plt.show()

In [11]:
img = visualtorch.graph_view(model, input_shape=INPUT_SHAPE)

plt.axis("off")
plt.tight_layout()
plt.imshow(img)
plt.show()

KeyboardInterrupt: 

## 3. Module tree + receptive field

`visualtorch.graph_view` does **not** work on this model at all (independent of depth): the forward returns a 6-tuple with `None` entries for inactive heads, and the autograd tracer crashes on `None` (`'NoneType' object has no attribute 'grad_fn'`). Use the module tree and the model's own `receptive_field_ms()` instead.

In [6]:
# Printed module tree (always works, no tracing) + receptive field.
print(model)

if hasattr(model, 'receptive_field_ms'):
    print(f"\nReceptive field: {model.receptive_field_ms():.0f} ms")

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")

VocalCoachTCN(
  (input_proj): Conv1d(40, 256, kernel_size=(1,), stride=(1,))
  (blocks): ModuleList(
    (0): TCNBlock(
      (conv): Conv1d(256, 256, kernel_size=(3,), stride=(1,))
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (act): GELU(approximate='none')
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TCNBlock(
      (conv): Conv1d(256, 256, kernel_size=(3,), stride=(1,), dilation=(2,))
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (act): GELU(approximate='none')
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (2): TCNBlock(
      (conv): Conv1d(256, 256, kernel_size=(3,), stride=(1,), dilation=(4,))
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (act): GELU(approximate='none')
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (3): TCNBlock(
      (conv): Conv1d(256, 256, kernel_size=(3,), stride=(1,), dilation=(8,))
      (norm): Layer